# 01 — HMEQ data stage

Cleans the raw file once, freezes the split, draws the shared bootstrap matrix, writes the observation flags (`blank_field`, `structural_none`, `unseen_level`, `partially_observed`) and Table 1's inputs. Every count is read off the frozen artifacts; a difference from the expected counts is reported, never adjusted.


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path


def repo_root(start: Path | None = None) -> Path:
    """The qmulp checkout this notebook lives in."""
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src" / "config.py").is_file() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("run this notebook from inside the qmulp checkout")


_root = repo_root()
sys.path.insert(0, str(_root / 'src'))
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import polars as pl
from IPython.display import display

from config import DEFAULT_CONFIG
from fingerprint import resolve_config, sha256_path
from paths import ROOT

pl.Config.set_tbl_rows(80)
pl.Config.set_tbl_cols(40)

# The frozen configuration this notebook runs with, echoed so the executed
# notebook records it. No absolute path is printed: artifacts are listed
# relative to the repository root.
config, smoke = resolve_config(DEFAULT_CONFIG)
DATASET = "hmeq"
print("dataset:", DATASET, "| smoke:", smoke)
print(json.dumps(config.as_dict(), indent=1, default=str))


## Run the stage


In [ ]:
from stages import data

result = data.run(DATASET, config)
display(result["rows"].to_pandas())
print("differences from the expected counts:", result["notes"] or "none")


## Artifacts


In [ ]:
for path in result["artifacts"]:
    print(f"{sha256_path(path)}  {Path(path).resolve().relative_to(ROOT)}")
